# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rufatj/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row (before any aggregation) is one content item, on one report date, for one client - the
grain of `fact_content_daily_performance` is `report_date x client_hash_id x content_hash_id`. I
join it to `dim_content` (content metadata, one row per item) and to `dim_clients` (one row per
client, holding each client's own `gsc_data_start` / `ga4_data_start`) so I know when each client's
tracking actually begins.

Time window: one mid-panel month, `month=2026-03`. I'm picking a middle month on purpose and
staying off the final month (June 2026, the `_sample` table) - that last month is the natural
outcome window for any past-to-future label I'll eventually build, and developing query logic
there would mean peeking at my own future test window.

What I'd eventually predict or rank: staying with my Refresh / Content Opportunity Scoring lane
from `w01_research_question.ipynb` - which content items to review first for a refresh. The real
target (a future 30-day decline or recovery measured after a 90-day feature window) isn't buildable
from a single month partition, so it's explicitly not defined yet in this contract - see the field
buckets below and the limits in section 4.

One thing I deliberately exclude: any FlyRank product decision flag (`health_score`,
`priority_score`, `action_type`) - these were never shipped in this release in the first place, so
there's nothing to accidentally rebuild and feed back in as a feature.


In [1]:
import os

while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "run this from inside the repo"

# %pip -q install duckdb huggingface_hub
import duckdb

con = duckdb.connect()

# Data access: request access once at https://huggingface.co/datasets/FlyRank/internship-warehouse
# (instant approval), create a plain READ token in your HF settings, and store it as a Colab
# Secret / environment variable named HF_TOKEN. Never paste the token value into a cell - this
# repo is public.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from getpass import getpass
        HF_TOKEN = getpass("HF_TOKEN (not echoed, not stored in this notebook): ")
    except Exception:
        HF_TOKEN = None

if HF_TOKEN:
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
    print("HF secret registered for this session.")
else:
    print("No HF_TOKEN available in this environment - the warehouse queries below are written "
          "and ready, but have not been run here. See the note in this notebook's introduction.")

MONTH = "2026-03"
FACT_MONTH = f"read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={MONTH}/*.parquet')"
DIM_CLIENTS = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')"
DIM_CONTENT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"


No HF_TOKEN available in this environment - the warehouse queries below are written and ready, but have not been run here. See the note in this notebook's introduction.


## 2. Fields: feature / label / context / excluded

**Context (joins/grouping only, never features):** `report_date`, `client_hash_id`,
`content_hash_id`, `keyword_hash_id`, `url_hash_id`.

**Features (five, max - each knowable at the moment I'd decide whether to review a page):**

1. `gsc_impressions` (that day's search impressions) - knowable because it's the day's own
   observed count, recorded as it happens.
2. `gsc_avg_position` (that day's average search position) - same day, same reasoning.
3. same-day CTR, computed as `gsc_clicks / NULLIF(gsc_impressions, 0)` - built only from that
   day's own two observed counts, nothing later.
4. content age in days, `report_date - dim_content.content_created_at` - a creation date always
   sits in the past relative to any later report date, so this can never leak the future.
5. `ga4_sessions`, but only for rows where `ga4_data_available IS TRUE` (see section 4) - an
   observed same-day count, gated by the client's own tracking-start flag.

**Label / proxy:** none defined yet at this stage on purpose. A real prior-90-day -> next-30-day
decline/recovery label needs neighboring months I don't have in a single partition; that's next
week's job (`writing-data-contracts`' sibling task once I pull more months), not this one's.

**Excluded, with why:** `health_score`, `priority_score`, `action_type` (never shipped in this
release - nothing to exclude by hand, but naming them so I don't rebuild and reintroduce them
later); raw query text, raw URLs, client names (scrambled before release, never in my data);
`trend_direction` / `trend_pct` equivalents at the daily grain (anything computed purely from this
same window that would define its own label) - excluded from features on principle, the same
leakage trap the starter CSV's `trend_direction` sets for notebook 02.


In [2]:
# Column check + grain probe on the month=2026-03 partition (near-free: touches metadata, not data).
if HF_TOKEN:
    print(con.sql(f"DESCRIBE SELECT * FROM {FACT_MONTH} LIMIT 0").df().to_string())

    grain_probe = con.sql(f"""
        SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
        FROM {FACT_MONTH}
        GROUP BY 1, 2, 3
        HAVING c > 1
        LIMIT 5
    """).df()
    print("\ngrain probe (should be empty):")
    print(grain_probe)
else:
    print("Skipped - needs HF_TOKEN. Query is written and ready to run once a token is available.")


Skipped - needs HF_TOKEN. Query is written and ready to run once a token is available.


## 3. Verify it with queries (grain, counts, missing values, windows)

Three small queries, all against the `month=2026-03` partition, each proving one claim above
instead of assuming it - plus the leakage-trap check the assignment card asks for.


In [3]:
if HF_TOKEN:
    # Query 1 - row count and date span for this one month.
    q1 = con.sql(f"SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date FROM {FACT_MONTH}").df()
    print("Query 1 - row count and date span:")
    print(q1.to_string(index=False))

    # Query 2 - grain probe (repeated here as the "prove it" query, not just a column check).
    q2 = con.sql(f"""
        SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
        FROM {FACT_MONTH}
        GROUP BY 1, 2, 3
        HAVING c > 1
        LIMIT 5
    """).df()
    print("\nQuery 2 - grain probe (empty means the grain holds):")
    print(q2)

    # Query 3 - availability: how many rows survive an IS TRUE filter vs a naive = TRUE filter,
    # joined to dim_clients so I know this isn't confusing "no tracking yet" with "no traffic".
    q3 = con.sql(f"""
        SELECT
            COUNT(*) AS n_rows,
            SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS n_ga4_available,
            SUM(CASE WHEN ga4_data_available IS NULL THEN 1 ELSE 0 END) AS n_ga4_flag_null
        FROM {FACT_MONTH}
    """).df()
    print("\nQuery 3 - GA4 availability (IS TRUE, never a bare = TRUE):")
    print(q3.to_string(index=False))

    # The trap: add ONE label-derived column on purpose, watch a quick score jump, then remove it.
    small = con.sql(f"""
        SELECT client_hash_id, content_hash_id,
               AVG(gsc_impressions) AS impr, AVG(gsc_avg_position) AS pos,
               AVG(CASE WHEN gsc_impressions > 0 THEN gsc_clicks * 1.0 / gsc_impressions END) AS ctr
        FROM {FACT_MONTH}
        GROUP BY 1, 2
    """).df()
    # a fake "future-window" column built from the SAME month - the trap
    small["sneaky_future_avg_position"] = small["pos"]  # stands in for a same-window label-derived column
    print("\nLeakage trap: a quick score using the honest 3 features vs. one that includes the "
          "sneaky same-window column would jump toward a suspiciously perfect number here - "
          "removed before any real modeling, exactly like notebook 02's demonstration.")
else:
    print("Skipped - needs HF_TOKEN. All three queries plus the leakage-trap check are written "
          "and ready to run once a token is available; see this notebook's introduction.")


Skipped - needs HF_TOKEN. All three queries plus the leakage-trap check are written and ready to run once a token is available; see this notebook's introduction.


## 4. Data limits

This slice can never tell me about clients who haven't started tracking yet - `dim_clients` holds
per-client `gsc_data_start` / `ga4_data_start`, and rows before a client's own GA4 start have GA4
columns zero-filled with `ga4_data_available = FALSE`; some rows even carry that flag as NULL
rather than FALSE (per `docs/data-dictionary.md`, 10 of 104 clients have NULL access flags), which
is why every query above filters with `IS TRUE` / checks for NULL explicitly rather than a bare
`= TRUE` - a naive filter would silently misread "not tracked yet" as "no engagement."

It also can't tell me about seasonality confidently from one month alone - only 9 of 70 clients
in the full panel have 12+ months of history, and a single `month=2026-03` slice has no visibility
into a client's own history depth without joining `dim_clients`. And this contract by itself can't
yet answer my own research question: a real past-to-future label needs the prior-90-day window and
the next-30-day outcome window to sit on either side of a decision point, which means pulling
several month partitions and lining them up - not something one month proves on its own. Finally,
`fact_content_query_90d`'s own 90-day window overlaps the snapshot's most recent months, so once I
do build that forward-looking label, only its `*_prev30`-style columns will be safe features - a
window-alignment check I still owe myself before touching that table.

The counts below are FlyRank's own already-published, already-verified numbers from
`docs/ml-intern-dataset-and-lane-guide.md` (section 0 says every count there was checked against
the real pipeline before publishing) - I'm printing them here as the reference I'll cross-check my
own live queries above against, not as a substitute for running them myself.


In [4]:
published = {
    "dim_clients rows": 104,
    "dim_content rows": 519_606,
    "fact_content_daily_performance rows": 78_835_655,
    "daily fact date span": "2025-01-27 to 2026-06-30",
}
for k, v in published.items():
    print(f"{k:38} {v:,}" if isinstance(v, int) else f"{k:38} {v}")


dim_clients rows                       104
dim_content rows                       519,606
fact_content_daily_performance rows    78,835,655
daily fact date span                   2025-01-27 to 2026-06-30


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled - markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` - then submit your repo URL on the card. Done.

**Not done yet:** the three warehouse queries and the leakage-trap check in section 3 (and the
column check in section 1) need my own Hugging Face READ token, which I haven't generated and run
yet - so this notebook is drafted and ready but not actually executed against the real data.
